In [1]:
!pip install afinn

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for afinn: filename=afinn-0.1-py3-none-any.whl size=53479 sha256=654a69e50b9ca2b37c4779d5ae5eb2d29d6caeae9bcce0953128fd97943b60e8
  Stored in directory: c:\users\rkim8\appdata\local\pip\cache\wheels\ee\d3\a0\f9255ebac29886acb1c28b35b37523f6399677fa06be379f25
Successfully built afinn


In [1]:
from afinn import Afinn
afinn = Afinn(language='en')

In [15]:
import pandas as pd
import numpy as np
data = pd.read_csv("dataFolder/train.csv", encoding="latin-1")
data = data.dropna(subset=["text"])

data[["text", "sentiment"]].head()

,text,sentiment
0,"I`d have responded, if I were going",neutral
1,Sooo SAD I will miss you here in San Diego!!!,negative
2,my boss is bullying me...,negative
3,what interview! leave me alone,negative
4,"Sons of ****, why couldn`t they put them on t...",negative


In [16]:
data["text"].shape

(27480,)

In [17]:
scores=[]
for i in data["text"]:
    scores.append(afinn.score(i))

data["affin score"] = scores

AFINN handles preprocessing and uses a lexicon based approach to do sentiment analysis (similar to Vader, Sentiwordnet, etc)

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²),affin score
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60,0.0
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105,-4.0
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18,-2.0
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164,-3.0
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26,0.0
...,...,...,...,...,...,...,...,...,...,...,...
27476,4eac33d1c0,wish we could come see u on Denver husband l...,d lost,negative,night,31-45,Ghana,31072940,227540.0,137,-2.0
27477,4f4c4fc327,I`ve wondered about rake to. The client has ...,", don`t force",negative,morning,46-60,Greece,10423054,128900.0,81,1.0
27478,f67aae2310,Yay good for both of you. Enjoy the break - y...,Yay good for both of you.,positive,noon,60-70,Grenada,112523,340.0,331,7.0
27479,ed167662a5,But it was worth it ****.,But it was worth it ****.,positive,night,70-100,Guatemala,17915568,107160.0,167,2.0


In [24]:
def afinn_to_sentiment(score):
    if score > 0:
        return 2
    elif score < 0:
        return 0
    else:
        return 1

def sentiment_to_score(sentiment):
    if sentiment == "positive":
        return 2
    elif sentiment == "negative":
        return 0
    else:
        return 1


data["afinn_pred"] = data["affin score"].apply(afinn_to_sentiment)
data["sentiment"] = data["sentiment"].apply(sentiment_to_score)
data[["text", "sentiment", "afinn_pred"]].head(10)

,text,sentiment,afinn_pred
0,"I`d have responded, if I were going",1,1
1,Sooo SAD I will miss you here in San Diego!!!,1,0
2,my boss is bullying me...,1,0
3,what interview! leave me alone,1,0
4,"Sons of ****, why couldn`t they put them on t...",1,1
5,http://www.dothebouncy.com/smf - some shameles...,1,2
6,2am feedings for the baby are fun when he is a...,1,2
7,Soooo high,1,1
8,Both of you,1,1
9,Journey!? Wow... u just became cooler. hehe....,1,2


In [25]:
def evaluate(true, pred):
    correct = 0
    total = 0

    for i in range(len(true)):
        total += 1
        if true[i] == pred[i]:
            correct += 1

    accuracy = correct / total
    print(f"Correct: {correct}/{total}")
    print(f"Accuracy: {accuracy:.2%}")

evaluate(list(data["sentiment"]), list(data["afinn_pred"]))

Correct: 8539/27480
Accuracy: 31.07%


Afinn has a very small and basic lexicon, so its pretty much trash. AFINN only has around 3,000 words in its lexicon. So anything outside that list scores 0